In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp

# Konfigurasi Klaster (Urutan: NGC 2516, M45, M44, M67, NGC 188)
clusters = [
    {"name": "NGC 2516", "file": "NGC2516_Members_Final.csv", "xlim": 100},
    {"name": "M45 (Pleiades)", "file": "M45_Members_Final.csv", "xlim": 180},
    {"name": "M44 (Praesepe)", "file": "M44_Members_Final.csv", "xlim": 150},
    {"name": "M67", "file": "M67_Members_Final.csv", "xlim": 60},
    {"name": "NGC 188", "file": "NGC188_Members_Final.csv", "xlim": 60}
]

# Setup Figure (2 Baris, 5 Kolom)
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(24, 10), constrained_layout=True)

# Loop untuk setiap klaster
for i, cluster in enumerate(clusters):
    filename = cluster["file"]
    name = cluster["name"]
    xlim_val = cluster["xlim"]

    # Load Data
    try:
        df = pd.read_csv(filename)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue

    # ---------------------------------------------------------
    # ROW 1: CMD COLORED BY RUWE
    # ---------------------------------------------------------
    ax_cmd = axes[0, i]

    # Data Cleaning untuk CMD
    cols_cmd = ['bp_rp', 'phot_g_mean_mag', 'ruwe']
    for col in cols_cmd:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df_cmd = df.dropna(subset=cols_cmd)

    # Sort biar merah di atas
    df_sorted = df_cmd.sort_values('ruwe')

    # Scatter Plot
    sc = ax_cmd.scatter(df_sorted['bp_rp'], df_sorted['phot_g_mean_mag'],
                        s=2, c=df_sorted['ruwe'], cmap='turbo', vmin=0.8, vmax=1.4, alpha=0.8)

    # Estetika CMD
    ax_cmd.invert_yaxis()
    ax_cmd.set_title(name, fontsize=14)
    ax_cmd.set_xlabel("Color ($G_{BP} - G_{RP}$)")
    if i == 0:
        ax_cmd.set_ylabel("Abs. Magnitude ($G$)")
    else:
        ax_cmd.set_ylabel("") # Biar gak penuh tulisan

    ax_cmd.grid(True, alpha=0.3)

    # Colorbar (Kecil di dalam atau di samping)
    cbar = plt.colorbar(sc, ax=ax_cmd, pad=0.02)
    cbar.set_label('RUWE', fontsize=8)
    cbar.ax.tick_params(labelsize=8)

    # ---------------------------------------------------------
    # ROW 2: CDF (MASS SEGREGATION)
    # ---------------------------------------------------------
    ax_cdf = axes[1, i]

    # Data Cleaning untuk CDF
    cols_cdf = ['ra', 'dec', 'ruwe']
    # Re-use df, ensure cols are numeric
    for col in cols_cdf:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df_cdf = df.dropna(subset=cols_cdf)

    # Hitung Pusat & Jarak
    center_ra = df_cdf['ra'].mean()
    center_dec = df_cdf['dec'].mean()

    d_ra = (df_cdf['ra'] - center_ra) * np.cos(np.radians(center_dec))
    d_dec = df_cdf['dec'] - center_dec
    df_cdf['radius_deg'] = np.sqrt(d_ra**2 + d_dec**2)
    df_cdf['radius_arcmin'] = df_cdf['radius_deg'] * 60

    # Bagi Populasi
    single_stars = df_cdf[df_cdf['ruwe'] < 1.1]
    binary_stars = df_cdf[df_cdf['ruwe'] > 1.2]

    # KS-Test
    if len(single_stars) > 0 and len(binary_stars) > 0:
        stat, p_value = ks_2samp(single_stars['radius_arcmin'], binary_stars['radius_arcmin'])
        title_text = f"p-value: {p_value:.2e}"
    else:
        p_value = 1.0
        title_text = "Insufficient Data"

    # Plot CDF
    ax_cdf.hist(single_stars['radius_arcmin'], bins=1000, density=True, cumulative=True,
                histtype='step', color='blue', linewidth=2, label='Single')
    ax_cdf.hist(binary_stars['radius_arcmin'], bins=1000, density=True, cumulative=True,
                histtype='step', color='red', linewidth=2, label='Binary')

    # Estetika CDF
    ax_cdf.set_title(title_text, fontsize=12, color='black')
    ax_cdf.set_xlabel('Dist. (arcmin)')
    ax_cdf.set_xlim(0, xlim_val)
    ax_cdf.grid(True, alpha=0.3)

    if i == 0:
        ax_cdf.set_ylabel('Cumul. Fraction')
        ax_cdf.legend(loc='lower right', fontsize=8)
    else:
        ax_cdf.set_ylabel("")


plt.savefig('evolution_grid_final.png', dpi=300)
plt.show()